In [1]:
import os
from pathlib import Path

import altair as alt
import attaviz
import pandas as pd
from IPython.display import display

attaviz.enable()


def find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from this notebook's directory until we find the project root."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
CHART_WIDTH = 600
CHART_HEIGHT = 300

# Tiempos de Viaje para ODs de Interés

Recolectamos tiempos de viaje y distancias recorridas entre los orígenes y destinos definidos para las líneas BRT. Los datos fueron recolectados entre el 27 de julio y el 1 de septiembre de 2026, cada 15 minutos. Durante la recolección, se perdieron algunas consultas por problemas técnicos que estamos trabajando en resolver.
Luego de recolectar los datos, agrupamos los tiempos y distancias a nivel horario y calculamos la media para ambas variables. Finalmente, generamos los gráficos que se muestran a continuación.

Observaciones:

- Para algunas horas del día, el par Ciudad Vieja - Zona América tiene distancias de viaje de 40 km. En Google, notamos que para ciertas horas del día puede ser más conveniente tomar caminos significativamente más largos. Nota: confirmar con alguien del lugar.
- Comparamos los tiempos de viaje contra los de Google y nuestra media se encuentra entre el tiempo de viaje mínimo y máximo informado por ellos. Nota: confirmar con alguien del lugar.
- En el par Av. 18 de Julio - Ciudad del Plata, el tiempo de viaje sube entre las 3 y las 4 AM. Es probable que se deba a que Mapbox tiene pocos datos de tráfico en tiempo real a esa hora y recurre a perfiles históricos, que pueden ser menos confiables. Nota: confirmar con alguien del lugar.

In [2]:
results_path = DATA_DIR / "travel-time"
files = sorted(results_path.glob("*.csv"))
aggregate_path = DATA_DIR / "travel_times_brt_mapbox.parquet"

if aggregate_path.exists():
    df_combined = pd.read_parquet(aggregate_path)
else:
    raw_frames = []
    for file in files:
        timestamp_utc = os.path.basename(file).replace(".csv", "")[-12:]
        column_names = {
            "od_id": "od_id",
            "od_name": "od_name",
            "origin": "origin",
            "destination": "destination",
            f"tt_{timestamp_utc}": "tt",
            f"dist_{timestamp_utc}": "dist",
            f"flipped_tt_{timestamp_utc}": "flipped_tt",
            f"flipped_dist_{timestamp_utc}": "flipped_dist",
        }
        numeric_dtypes = {
            "od_id": "int64",
            f"tt_{timestamp_utc}": "float64",
            f"dist_{timestamp_utc}": "float64",
            f"flipped_tt_{timestamp_utc}": "float64",
            f"flipped_dist_{timestamp_utc}": "float64",
        }
        raw_frames.append(
            pd.read_csv(
                file,
                usecols=list(column_names),
                dtype=numeric_dtypes,
            )
            .rename(columns=column_names)
            .assign(timestamp_utc=timestamp_utc)
        )

    raw = pd.concat(raw_frames, ignore_index=True)
    local_datetimes = pd.to_datetime(
        raw["timestamp_utc"], format="%Y%m%d%H%M", utc=True
    ).dt.tz_convert("America/Montevideo")
    metadata = pd.DataFrame(
        {
            "year": local_datetimes.dt.year,
            "month": local_datetimes.dt.month,
            "day": local_datetimes.dt.day,
            "hour": local_datetimes.dt.hour,
            "minute": local_datetimes.dt.minute,
            "date": local_datetimes.dt.strftime("%Y-%m-%d"),
            "datetime": local_datetimes.dt.strftime("%Y%m%d%H%M"),
        }
    )

    base = raw[["od_id", "od_name"]]
    forward = base.assign(
        origin=raw["origin"].to_numpy(),
        destination=raw["destination"].to_numpy(),
        tt=raw["tt"].to_numpy(),
        dist=raw["dist"].to_numpy(),
    )
    flipped = base.assign(
        origin=raw["destination"].to_numpy(),
        destination=raw["origin"].to_numpy(),
        tt=raw["flipped_tt"].to_numpy(),
        dist=raw["flipped_dist"].to_numpy(),
    )
    directions = pd.concat([forward, flipped], ignore_index=True)
    metadata = pd.concat([metadata, metadata], ignore_index=True)
    df_combined = pd.concat([directions, metadata], axis=1)

    df_combined["time"] = pd.to_datetime(
        df_combined["hour"].astype("str") + ":" + df_combined["minute"].astype("str"),
        format="%H:%M",
    )
    df_combined["tt"] = df_combined["tt"] / 60
    df_combined["dist"] = (
        df_combined["dist"] / 1000
    )  # Convert dist from meters to kilometers
    df_combined["weekday"] = pd.to_datetime(df_combined["date"]).dt.day_name()
    df_combined["is_weekend"] = df_combined["weekday"].isin(["Saturday", "Sunday"])
    df_combined.sort_values(by=["od_id", "datetime"], inplace=True)
    df_combined.to_parquet(aggregate_path, index=False)

print(df_combined.shape)

(20742, 16)


In [3]:
def create_plots(od_, variable, title_prefix, ylabel):
    """Create an interactive Altair chart for one OD pair and measure."""
    daily_profiles = (
        od_.groupby(
            ["date", "hour", "origin", "destination"],
            as_index=False,
            observed=True,
        )[variable]
        .mean()
        .rename(columns={variable: "value"})
        .assign(series_type="Daily")
    )
    mean_profiles = (
        od_.groupby(
            ["hour", "origin", "destination"],
            as_index=False,
            observed=True,
        )[variable]
        .mean()
        .rename(columns={variable: "value"})
        .assign(series_type="Average")
    )
    plot_data = pd.concat([daily_profiles, mean_profiles], ignore_index=True)
    plot_data["direction"] = plot_data["origin"] + " → " + plot_data["destination"]

    direction_order = plot_data["direction"].drop_duplicates().tolist()
    color_scale = alt.Scale(
        domain=direction_order,
        range=attaviz.CATEGORICAL[: len(direction_order)],
    )
    direction_selection = alt.selection_point(
        fields=["direction"],
        bind="legend",
        name="direction_select",
    )

    x = alt.X(
        "hour:Q",
        title="Hour of day",
        scale=alt.Scale(domain=[0, 23]),
        axis=alt.Axis(
            values=list(range(0, 24, 2)),
            labelExpr="datum.value + ':00'",
        ),
    )
    color = alt.Color(
        "direction:N",
        title="Direction",
        sort=direction_order,
        scale=color_scale,
    )
    daily_tooltip = [
        alt.Tooltip("direction:N", title="Direction"),
        alt.Tooltip("date:N", title="Date"),
        alt.Tooltip("hour:Q", title="Hour", format="02d"),
        alt.Tooltip("value:Q", title=ylabel, format=".1f"),
    ]
    mean_tooltip = [
        alt.Tooltip("direction:N", title="Direction"),
        alt.Tooltip("series_type:N", title="Profile"),
        alt.Tooltip("hour:Q", title="Hour", format="02d"),
        alt.Tooltip("value:Q", title=ylabel, format=".1f"),
    ]
    y = alt.Y("value:Q", title=ylabel)

    daily_lines = (
        alt.Chart(plot_data)
        .transform_filter(alt.datum.series_type == "Daily")
        .mark_line(size=1)
        .encode(
            x=x,
            y=y,
            color=color,
            detail=alt.Detail("date:N"),
            opacity=alt.condition(
                direction_selection, alt.value(0.20), alt.value(0.05)
            ),
            tooltip=daily_tooltip,
        )
    )
    mean_lines = (
        alt.Chart(plot_data)
        .transform_filter(alt.datum.series_type == "Average")
        .mark_line()
        .encode(
            x=x,
            y=y,
            color=color,
            opacity=alt.condition(direction_selection, alt.value(1), alt.value(0.1)),
            tooltip=mean_tooltip,
        )
    )

    od_name = od_["od_name"].iloc[0]
    chart = (
        (daily_lines + mean_lines)
        .add_params(direction_selection)
        .properties(
            title=alt.TitleParams(
                text=f"{title_prefix} in {od_name}",
                subtitle=[
                    "Thin lines show daily weekday profiles and thick lines show the mean.",
                    "Select directions in the legend.",
                ],
            )
        )
    )
    return attaviz.configure_size(chart, width=CHART_WIDTH, height=CHART_HEIGHT)

## Ciudad Vieja - Zonamerica (Días de semana)

In [4]:
od_ = df_combined.copy().query("od_id == 0 and is_weekend == False")

display(
    create_plots(od_, "tt", "Travel time (in minutes)", "Travel time by hour of day")
)
display(create_plots(od_, "dist", "Distance (in km)", "Distance by hour of day"))

alt.LayerChart(...)

alt.LayerChart(...)

## Ciudad Vieja - El Pinar (Días de semana)

In [5]:
od_ = df_combined.copy().query("od_id == 1 and is_weekend == False")

display(
    create_plots(od_, "tt", "Travel time (in minutes)", "Travel time by hour of day")
)
display(create_plots(od_, "dist", "Distance (in km)", "Distance by hour of day"))

alt.LayerChart(...)

alt.LayerChart(...)

## Av. 18 de Julio - Ciudad del Plata (Días de semana)

In [6]:
od_ = df_combined.copy().query("od_id == 2 and is_weekend == False")

display(
    create_plots(od_, "tt", "Travel time (in minutes)", "Travel time by hour of day")
)
display(create_plots(od_, "dist", "Distance (in km)", "Distance by hour of day"))


alt.LayerChart(...)

alt.LayerChart(...)